# Lab 6：集群混合并行训练

## 实验目标

- 在 ModelArts 的单机 4 卡 Ascend 910B 实例上配置 MindSpeed-LLM 训练环境。
- 将 Qwen2.5-7B 的 HuggingFace 权重转换为 TP=2、PP=2 的 Megatron 权重分片，并完成 Alpaca 数据预处理。
- 启动 Qwen2.5-7B 多卡训练，从日志中读取 loss、吞吐和单步耗时。


## 实验环境

| 项目 | 配置 |
| --- | --- |
| 平台 | 华为云 ModelArts Notebook，浏览器内使用 JupyterLab |
| 规格 | 单机 4×Ascend 910B，单卡 64GB HBM2e |
| 镜像 | mindspeed_llm_2.2.0 公共镜像，AI 引擎选择 MindSpeed-LLM |
| Python | 3.11.10 |
| 软件 | PyTorch 2.7.1、torch_npu、MindSpeed-LLM 2.2.0 |

所有实验文件保存在 /home/ma-user/work/。这是实例的 EVS 持久化目录。


## 实验原理

Qwen2.5-7B 使用 TP=2、PP=2、DP=1 的混合并行配置。张量并行将层内权重分到两张卡；流水线并行将 28 层模型分为两个 stage；数据并行度为 1。

本实验的全局批次为 GBS=64，微批次为 MBS=1，因此梯度累积步数为 GAS = 64 / (1 × 1) = 64。HuggingFace 格式的完整模型权重会按 TP 和 PP 转换为 Megatron 权重分片，训练数据则从 parquet 转为 .bin 和 .idx 文件。


## 实验流程

以下单元按顺序执行。训练脚本在 JupyterLab 终端中运行。


In [ ]:
# 屏蔽不必要的警告（保持输出简洁）
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONWARNINGS'] = 'ignore'          # 传递给后续 Python 子进程
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'

# 全局路径与并行配置
from pathlib import Path

WORK_DIR = Path('/home/ma-user/work/l06_workspace')   # 所有实验文件的根目录
WORK_DIR.mkdir(parents=True, exist_ok=True)

MSLLM_ROOT = Path('/home/ma-user/MA_Turbo/src/open_source/MindSpeed-LLM')
HF_DIR      = WORK_DIR / 'Qwen2.5-7B'         # HuggingFace 权重
RAW_DATA    = WORK_DIR / 'alpaca.parquet'     # 原始训练数据
DATA_PREF   = WORK_DIR / 'alpaca'             # 预处理输出前缀
TOTAL       = 4                    # 总卡数
TP, PP      = 2, 2                 # 张量并行 / 流水线并行（4 卡 = TP2 x PP2，DP=1）
# 显存依据：每卡约 1.9B 参数，模型+优化器约 38GB，加激活与运行开销约 42GB / 64GB，余量充足
MCORE_DIR   = WORK_DIR / f'qwen_mcore_tp{TP}_pp{PP}'
print('工作目录：', WORK_DIR)


### 1. 配置混合并行参数

4 张卡按 TP=2、PP=2、DP=1 使用：

- TP=2：每层权重分成两份，两张卡分别计算。
- PP=2：28 层模型分为两个 stage，各包含 14 层。流水线气泡占比约为 (PP - 1) / (GAS + PP - 1) = 1 / 65。
- GBS=64、MBS=1、DP=1，因此 GAS=64。每张卡累积 64 个微批次后更新一次参数。


### 2. 下载模型权重


In [ ]:
# 安装 modelscope（ModelArts 镜像未预装）
!pip install modelscope -q

In [ ]:
# 下载 Qwen2.5-7B 权重（约15GB）
# 先检查是否已下载完整，完整则跳过
_need_download = True
if (HF_DIR / 'config.json').is_file():
    _safetensors = list(HF_DIR.glob('*.safetensors'))
    if len(_safetensors) >= 4:
        _need_download = False
        print('权重已存在（%d 个 safetensors），跳过下载' % len(_safetensors))

if _need_download:
    import logging
    import threading
    import time
    from IPython.display import clear_output

    # 压制 SDK 自身的进度输出（tqdm 在 JupyterLab 中每次刷新都会新增一行，会刷屏）
    os.environ['MODELSCOPE_LOG_LEVEL'] = str(logging.ERROR)  # 必须数字字符串：modelscope 用 int() 解析，'ERROR' 会报错
    os.environ['MODELSCOPE_NO_DEPRECATION_WARNINGS'] = '1'
    os.environ['TQDM_DISABLE'] = '1'

    # 下载放后台线程，主线程每 5 秒统计目录实际大小，clear_output 单行刷新进度
    _err = []

    def _worker():
        try:
            from modelscope.hub.snapshot_download import snapshot_download
            snapshot_download('Qwen/Qwen2.5-7B', local_dir=str(HF_DIR))
        except Exception as e:
            _err.append(e)

    _t = threading.Thread(target=_worker)
    _t.start()

    _TARGET_GB = 15.2   # Qwen2.5-7B 权重约 15.2GB
    _start = time.time()
    while _t.is_alive():
        time.sleep(5)
        _size = sum(f.stat().st_size for f in HF_DIR.rglob('*') if f.is_file()) if HF_DIR.exists() else 0
        _pct = min(100.0, _size / 1e9 / _TARGET_GB * 100)
        _speed = (_size / 1e9) / max(1, time.time() - _start)
        clear_output(wait=True)   # 清掉旧输出只留这一行，进度实时且不堆积
        print('下载中：%.1f GB / 约 %.1f GB（%.0f%%，%.1f MB/s）'
              % (_size / 1e9, _TARGET_GB, _pct, _speed * 1024))
    _t.join()
    if _err:
        raise _err[0]

    clear_output(wait=True)
    print('下载完成：', HF_DIR)
    _final = list(HF_DIR.glob('*.safetensors'))
    print('safetensors 文件数：%d（应为 4）' % len(_final))

下载完成后，查看权重目录：


In [ ]:
# 查看下载的权重文件
!ls -lh {HF_DIR} | head -20

权重目录中包含以下文件：

- config.json：模型架构信息。
- *.safetensors：模型权重，共 4 个文件。
- tokenizer.json：分词器。

### 3. 下载训练数据


In [ ]:
# 下载 Alpaca 数据集（约 24MB，下载耗时可忽略）
# 主链接 huggingface.co（国内可能连不上），自动回退 hf-mirror 镜像
ALPACA_URLS = [
    'https://huggingface.co/datasets/tatsu-lab/alpaca/resolve/main/data/train-00000-of-00001-a09b74b3ef9c3b56.parquet',
    'https://hf-mirror.com/datasets/tatsu-lab/alpaca/resolve/main/data/train-00000-of-00001-a09b74b3ef9c3b56.parquet',
]
for _url in ALPACA_URLS:
    !wget -q -c --timeout=30 --tries=2 {_url} -O {RAW_DATA}
    if RAW_DATA.exists() and RAW_DATA.stat().st_size > 1024:
        print('下载成功：', _url.split('/')[2])
        break
!ls -lh {RAW_DATA} | head -20


### 4. 检查 NPU 与通信

先查看 NPU 状态并确认 torch_npu 可用，再执行 AllReduce 通信测试。


In [ ]:
# 查看所有 NPU 的状态（昇腾的硬件状态查看命令）
!npu-smi info

npu-smi info 会列出 NPU 编号、型号、显存占用和温度。本实验的 4 卡实例应显示 4 张卡，单卡显存为 64GB。


In [ ]:
# 确认 torch_npu 可用，统计 NPU 数量
import torch
import torch_npu
print('可用 NPU 数量：', torch.npu.device_count())

#### 验证 AllReduce 通信

通信测试让每张卡参与一次求和。4 张卡时，结果为 1 + 2 + 3 + 4 = 10。


In [ ]:
# 验证多卡通信（用 torch_npu 做 AllReduce）
n_npu = torch.npu.device_count()
print('当前 NPU 数量：', n_npu)

if n_npu < 2:
    print('只有 1 张卡，跳过通信测试（本实验用 4 卡实例，正常应显示 4 张）')
else:
    # 写一个临时验证脚本，用 torchrun 拉起
    check_script = WORK_DIR / 'hccl_check.py'
    check_script.write_text(
        "import os, torch, torch_npu\n"
        "import torch.distributed as dist\n"
        "local_rank = int(os.environ['LOCAL_RANK'])\n"
        "rank = int(os.environ['RANK'])\n"
        "world = int(os.environ['WORLD_SIZE'])\n"
        "torch.npu.set_device(local_rank)\n"
        "dist.init_process_group(backend='hccl')\n"
        "t = torch.ones(1024, 1024, device='npu:%d' % local_rank) * (rank + 1)\n"
        "dist.all_reduce(t, op=dist.ReduceOp.SUM)\n"
        "expected = sum(r + 1 for r in range(world))\n"
        "ok = torch.allclose(t, torch.ones(1024, 1024, device='npu:%d' % local_rank) * expected)\n"
        "print('rank %d/%d: AllReduce correct = %s' % (rank, world, 'OK' if ok else 'FAIL'), flush=True)\n"
        "dist.barrier()\n"
        "if rank == 0:\n"
        "    print('=== HCCL check %s ===' % ('PASSED' if ok else 'FAILED'), flush=True)\n"
        "dist.destroy_process_group()\n",
        encoding='utf-8',
    )
    cmd = ('source /usr/local/Ascend/ascend-toolkit/set_env.sh 2>/dev/null && '
           'export HCCL_WHITELIST_DISABLE=1 && '
           'export HCCL_CONNECT_TIMEOUT=7200 && '
           'torchrun --nproc_per_node %d --master_addr 127.0.0.1 --master_port 29500 %s' % (n_npu, check_script))
    print('执行通信测试...')
    !{cmd}

### 5. 转换模型权重

HuggingFace 权重是完整模型权重。训练前，需要按张量并行切分层内权重，并按流水线并行切分模型层，使每张卡加载对应分片。

<img src="images/l06_step2_weight_split.png" alt="权重切分示意图" width="560">

#### 转换参数

本实验使用 TP=2、PP=2。权重转换命令中的 --target-tensor-parallel-size 2 和 --target-pipeline-parallel-size 2 必须与训练配置相同。

#### 检查 MindSpeed-LLM 目录

转换脚本 convert_ckpt.py 位于 MindSpeed-LLM 仓库根目录。


In [ ]:
# 使用预装的 MindSpeed-LLM，并补齐 Megatron-LM 依赖
import subprocess, sys

# 使用镜像预装的 MindSpeed-LLM（固定路径）
print('MindSpeed-LLM 仓库:', MSLLM_ROOT)
assert (MSLLM_ROOT / 'convert_ckpt.py').is_file(), '预装 MindSpeed-LLM 不完整，请确认镜像为 mindspeed_llm_2.2.0'

# 把仓库根目录加入 Python 路径（megatron 要从这里加载）
if str(MSLLM_ROOT) not in sys.path:
    sys.path.insert(0, str(MSLLM_ROOT))

# 补齐 Megatron-LM 依赖（官方 install_guide 要求 core_v0.12.1）
try:
    import megatron
    print('megatron: 已就绪')
except ImportError:
    print('megatron 缺失，正在按官方流程安装...')
    # 官方安装流程：clone → checkout core_v0.12.1 → 拷贝 megatron 目录
    megatron_src = WORK_DIR / 'Megatron-LM'
    if not (megatron_src / 'megatron' / '__init__.py').is_file():
        if megatron_src.exists():
            import shutil; shutil.rmtree(megatron_src)
        subprocess.run(['git', 'clone', 'https://github.com/NVIDIA/Megatron-LM.git', str(megatron_src)], check=True, capture_output=True)
    subprocess.run(['git', '-C', str(megatron_src), 'checkout', 'core_v0.12.1'], check=True)
    # 拷贝到 MindSpeed-LLM 目录（官方要求）
    subprocess.run(['cp', '-r', str(megatron_src / 'megatron'), str(MSLLM_ROOT / 'megatron')], check=True)
    import megatron
    print('megatron: 安装完成')

# 验证关键脚本存在
print('convert_ckpt.py:', (MSLLM_ROOT / 'convert_ckpt.py').is_file())
print('preprocess_data.py:', (MSLLM_ROOT / 'preprocess_data.py').is_file())
print('pretrain_gpt.py:', (MSLLM_ROOT / 'pretrain_gpt.py').is_file())
print('环境就绪，可以继续下一步')

#### 执行权重转换


In [ ]:
# 执行权重转换（参数对齐官方 v2.2.0 的 ckpt_convert_qwen25_hf2mcore.sh）
# 转换脚本会打印全量配置 dump 和逐层进度（Megatron 侧 print，环境变量关不掉），
# 因此输出全部写入日志文件：成功只显示末尾 20 行，失败显示末尾 80 行（报错 traceback 在末尾）

%cd {MSLLM_ROOT}
!source /usr/local/Ascend/ascend-toolkit/set_env.sh 2>/dev/null && \
 export CUDA_DEVICE_MAX_CONNECTIONS=1 && \
 export PYTHONWARNINGS=ignore && \
 export TRANSFORMERS_VERBOSITY=error && \
 echo '开始权重转换（约 10 分钟），完整日志：{WORK_DIR}/convert_ckpt.log' && \
 if python -W ignore convert_ckpt.py \
    --use-mcore-models \
    --model-type GPT \
    --load-model-type hf \
    --save-model-type mg \
    --target-tensor-parallel-size {TP} \
    --target-pipeline-parallel-size {PP} \
    --add-qkv-bias \
    --load-dir {HF_DIR}/ \
    --save-dir {MCORE_DIR}/ \
    --tokenizer-model {HF_DIR}/tokenizer.json \
    --model-type-hf llama2 \
    --params-dtype bf16 \
    > {WORK_DIR}/convert_ckpt.log 2>&1; then \
    echo '===== 转换完成，日志末尾 20 行 ====='; tail -20 {WORK_DIR}/convert_ckpt.log; \
 else \
    echo '===== 转换失败，日志末尾 80 行 ====='; tail -80 {WORK_DIR}/convert_ckpt.log; \
 fi

#### 查看转换参数

| 参数 | 值 | 含义 |
| --- | --- | --- |
| --use-mcore-models | - | 使用 Megatron-Core 模型实现 |
| --target-tensor-parallel-size | 2 | 按 TP=2 切分层内权重 |
| --target-pipeline-parallel-size | 2 | 按 PP=2 切分模型层 |
| --add-qkv-bias | - | 保留 Qwen2.5 的 QKV 偏置 |
| --model-type-hf llama2 | - | 使用与 Qwen 架构兼容的转换规则 |
| --params-dtype bf16 | - | 以 bf16 保存权重 |

#### 查看转换结果


In [ ]:
# 转换后应该生成 mp_rank_* 目录，共 TP x PP = 4 个
!ls -la {MCORE_DIR}/ | head -20

### 6. 预处理训练数据

模型读取 token 序列。这里使用 MindSpeed 的 preprocess_data.py，将 Alpaca parquet 数据转为 Megatron 格式的 .bin 数据文件和 .idx 索引文件。


In [ ]:
# 数据预处理（preprocess_data.py 同样会打印 Megatron 配置 dump 与 tokenizer 加载日志，
# 输出写入日志文件：成功只显示末尾 15 行，失败显示末尾 80 行）

%cd {MSLLM_ROOT}
!source /usr/local/Ascend/ascend-toolkit/set_env.sh 2>/dev/null && \
 export PYTHONWARNINGS=ignore && \
 export TRANSFORMERS_VERBOSITY=error && \
 echo '开始数据预处理（约 5 分钟），完整日志：{WORK_DIR}/preprocess_data.log' && \
 if python -W ignore ./preprocess_data.py \
    --input {RAW_DATA} \
    --tokenizer-name-or-path {HF_DIR}/ \
    --output-prefix {DATA_PREF} \
    --tokenizer-type PretrainedFromHF \
    --workers 4 \
    --log-interval 1000 \
    > {WORK_DIR}/preprocess_data.log 2>&1; then \
    echo '===== 预处理完成，日志末尾 15 行 ====='; tail -15 {WORK_DIR}/preprocess_data.log; \
 else \
    echo '===== 预处理失败，日志末尾 80 行 ====='; tail -80 {WORK_DIR}/preprocess_data.log; \
 fi

In [ ]:
# 确认产物（应该看到 .bin 和 .idx 两个文件）
!ls -lh {DATA_PREF}_text_document.bin {DATA_PREF}_text_document.idx | head -20


### 7. 生成训练脚本

MindSpeed 使用 torchrun 启动训练。下表列出本实验脚本中的主要参数。

| 模块 | 参数 | 取值 |
| --- | --- | --- |
| 模型 | --num-layers、--hidden-size、--num-attention-heads | 28、3584、28 |
| 并行 | --tensor-model-parallel-size、--pipeline-model-parallel-size | TP=2、PP=2 |
| 数据 | --data-path、--seq-length、--micro-batch-size、--global-batch-size | alpaca_text_document、8192、1、64 |
| 训练 | --train-iters、--lr、--bf16 | 100、1.25e-6、开启 |
| 优化器 | --adam-beta1、--adam-beta2、--weight-decay、--clip-grad | 0.9、0.95、0.1、1.0 |

配置需满足以下关系：

1. NPU 总数可被 TP × PP × CP 整除：4 可被 2 × 2 × 1 整除。
2. 注意力头数可被 TP × CP 整除：28 可被 2 × 1 整除。
3. GBS / MBS 可被数据并行度整除：64 / 1 可被 DP=1 整除。


In [ ]:
# 生成训练启动脚本（参数对齐官方 pretrain_qwen25_7b_32k_ptd.sh）
SEQ_LEN = 8192
RECOMPUTE_LAYERS = 28 // PP   # PP=2 时每个 stage 14 层
OUT_DIR = WORK_DIR / f'output_tp{TP}_pp{PP}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_SH  = OUT_DIR / 'run_train.sh'
RUN_LOG = OUT_DIR / 'run_log'

script = rf'''#!/bin/bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh 2>/dev/null
export CUDA_DEVICE_MAX_CONNECTIONS=1
export HCCL_WHITELIST_DISABLE=1
export HCCL_CONNECT_TIMEOUT=7200
export PYTHONWARNINGS=ignore
export TRANSFORMERS_VERBOSITY=error
export TOKENIZERS_PARALLELISM=false

# train-iters 官方脚本为 2000（30 小时以上），此处设 100，约 1.5-2 小时，100 个 loss 点足以画出平滑下降曲线
torchrun --nproc_per_node {TOTAL} --master_addr localhost --master_port 6000 \
    pretrain_gpt.py \
    --use-mcore-models \
    --tensor-model-parallel-size {TP} \
    --pipeline-model-parallel-size {PP} \
    --sequence-parallel \
    --num-layers 28 \
    --hidden-size 3584 \
    --ffn-hidden-size 18944 \
    --num-attention-heads 28 \
    --max-position-embeddings {SEQ_LEN} \
    --seq-length {SEQ_LEN} \
    --disable-bias-linear \
    --add-qkv-bias \
    --group-query-attention \
    --num-query-groups 4 \
    --use-flash-attn \
    --swiglu \
    --use-fused-swiglu \
    --normalization RMSNorm \
    --norm-epsilon 1e-6 \
    --use-fused-rmsnorm \
    --position-embedding-type rope \
    --rotary-base 1000000 \
    --use-fused-rotary-pos-emb \
    --untie-embeddings-and-output-weights \
    --micro-batch-size 1 \
    --global-batch-size 64 \
    --make-vocab-size-divisible-by 1 \
    --padded-vocab-size 152064 \
    --tokenizer-type PretrainedFromHF \
    --tokenizer-name-or-path {HF_DIR} \
    --attention-dropout 0.0 \
    --hidden-dropout 0.0 \
    --train-iters 100 \
    --lr 1.25e-6 \
    --lr-decay-style cosine \
    --min-lr 1.25e-7 \
    --lr-warmup-fraction 0.01 \
    --init-method-std 0.01 \
    --weight-decay 0.1 \
    --clip-grad 1.0 \
    --adam-beta1 0.9 \
    --adam-beta2 0.95 \
    --initial-loss-scale 4096 \
    --no-gradient-accumulation-fusion \
    --no-masked-softmax-fusion \
    --attention-softmax-in-fp32 \
    --bf16 \
    --recompute-granularity full \
    --recompute-method block \
    --recompute-num-layers {RECOMPUTE_LAYERS} \
    --data-path {DATA_PREF}_text_document \
    --split 100,0,0 \
    --load {MCORE_DIR} \
    --no-load-optim \
    --no-load-rng \
    --save {OUT_DIR}/saved_checkpoints \
    --no-save-optim \
    --no-save-rng \
    --distributed-backend nccl \
    --log-interval 1 \
    --save-interval 2000 \
    --eval-interval 2000 \
    --eval-iters 0 \
    --log-throughput \
    2>&1 | tee {RUN_LOG}
'''
RUN_SH.write_text(script, encoding='utf-8')
print('训练脚本已生成：', RUN_SH)

#### 在终端启动训练

在 JupyterLab 中打开终端后执行：

    cd /home/ma-user/MA_Turbo/src/open_source/MindSpeed-LLM
    bash /home/ma-user/work/l06_workspace/output_tp2_pp2/run_train.sh

日志出现 iteration 1 和 lm loss 后，训练已经开始。


### 8. 解析训练日志

训练日志 run_log 记录每一步的训练状态。后续单元会提取以下字段：

| 字段 | 含义 |
| --- | --- |
| iteration | 当前步数 |
| lm loss | 当前步损失 |
| tokens/s/p | 每卡每秒处理的 token 数 |
| elapsed time | 单步耗时 |


In [ ]:
# 从日志提取 loss / 吞吐 / 耗时
# 正则宽容匹配：MindSpeed 日志继承 Megatron 格式，
# 吞吐字段为 throughput，耗时字段为 seconds per iteration
import re, math

if RUN_LOG.is_file():
    txt = RUN_LOG.read_text(encoding='utf-8', errors='replace')
    NUM = r'([-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)'
    losses  = [float(m.group(1)) for m in re.finditer(r'lm loss\s*[:=]\s*' + NUM, txt)]
    losses  = [x for x in losses if math.isfinite(x)]
    tputs   = [float(m.group(1)) for m in re.finditer(r'(?:throughput|tokens\s*/\s*s)[^0-9\n]*?' + NUM, txt, re.I)]
    elapsed = [float(m.group(1)) for m in re.finditer(r'(?:elapsed time(?:\s+per\s+iteration)?|seconds\s+per\s+iteration)[^0-9\n]*?' + NUM, txt, re.I)]
    print(f'提取到 {len(losses)} 步 loss')
    if losses:
        print(f'  首 step: lm loss = {losses[0]:.4f}')
        print(f'  末 step: lm loss = {losses[-1]:.4f}')
    if tputs:
        print(f'  平均吞吐: {sum(tputs)/len(tputs):.1f}')
    if elapsed:
        print(f'  平均单步: {sum(elapsed)/len(elapsed):.0f}')
else:
    losses, tputs, elapsed = [], [], []
    print('日志还没生成。请先在终端完成 Step 4 训练，然后回到这里。')



#### 绘制 loss 曲线

loss 曲线用于观察训练过程中的损失变化。


In [ ]:
# 画 loss 曲线
if losses:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 4))
    plt.plot(range(1, len(losses)+1), losses, color='#d6336c', lw=1.5, marker='o', ms=3)
    plt.xlabel('iteration')
    plt.ylabel('lm loss')
    trend = 'normal (descending)' if losses[-1] < losses[0] else 'ABNORMAL (check!)'
    plt.title(f'lm loss: {losses[0]:.3f} -> {losses[-1]:.3f} [{trend}]')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('暂无 loss 数据，训练完成后重新运行本步骤。')

## 实验总结

本实验完成了模型权重下载、混合并行权重转换、数据预处理、训练脚本生成和日志解析。权重转换与训练使用相同的 TP、PP 参数，预处理生成的 .bin、.idx 文件由训练脚本通过 --data-path 读取。

## 实验扩展

1. 为什么 TP×PP 必须 ≤ 总卡数？在 4 卡上设 TP=4 PP=2 会发生什么？

2. seq_length 从 8192 改成 32768，显存和速度会有什么变化？

3. loss 曲线先降后平，说明什么？如何继续优化？

## 参考答案

运行以下单元查看思考题答案：


In [ ]:
!cat answer/thought_questions.txt